In [ ]:
import sys, os, time, math
import numpy as np, yaml, cloudpickle
sys.path.append(os.path.abspath('..'))
from drone_env.drone_sim import RoomDroneEnv
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import pybullet as p

STAGE_TO_WATCH = 1  # 0=Hover 1=FaceIt 2=Scout 3=Navigator 4=Hunter ...
RENDER_STRIDE  = 10

config_path = os.path.abspath(os.path.join('..', 'configs', 'teacher_ppo.yaml'))
with open(config_path) as f:
    config = yaml.safe_load(f)
sc = config['stages'][f'stage_{STAGE_TO_WATCH}']

HOVER_ONLY = sc.get('hover_only', False)
FACE_ONLY  = sc.get('face_only', False)
RUN_NAME   = sc['run_name']
if FACE_ONLY:   rw = config['face_rewards']
elif HOVER_ONLY: rw = config['hover_rewards']
else:            rw = config['nav_rewards']

env_raw = RoomDroneEnv(gui=True,
    num_obstacles=sc['num_obstacles'], randomize_obstacles=sc['randomize_obstacles'],
    randomize_coins=sc['randomize_coins'], reward_weights=rw,
    hover_only=HOVER_ONLY, num_fixed_coins=sc.get('num_fixed_coins',4),
    fixed_spawn=sc.get('fixed_spawn',False), coin_spawn_radius=sc.get('coin_spawn_radius'),
    face_only=FACE_ONLY, face_spawn_radius=sc.get('face_spawn_radius',3.0),
    face_threshold=sc.get('face_threshold',0.95), face_consecutive_steps=sc.get('face_consecutive_steps',10))
env_vec = DummyVecEnv([lambda e=env_raw: e])

MODEL_DIR    = os.path.abspath(os.path.join('..','models',f'stage_{STAGE_TO_WATCH}',RUN_NAME))
model_path   = os.path.join(MODEL_DIR,'best_model')
vecnorm_path = f"{model_path}_vecnormalize.pkl"
print(f"[S{STAGE_TO_WATCH}|{RUN_NAME}] Waiting for {model_path}.zip ...")
while not (os.path.exists(f"{model_path}.zip") and os.path.exists(vecnorm_path)):
    print("  waiting..."); time.sleep(5)

env = VecNormalize.load(vecnorm_path, env_vec)
env.training = False; env.norm_reward = False
model = PPO.load(model_path, env=env)
obs = env.reset()
p.resetDebugVisualizerCamera(cameraDistance=6.0, cameraYaw=45, cameraPitch=-30, cameraTargetPosition=[0,0,1])
print(f"[{RUN_NAME}] Running. RENDER_STRIDE={RENDER_STRIDE}")

def _sphere(pos, r, color): vis=p.createVisualShape(p.GEOM_SPHERE,radius=r,rgbaColor=color); p.createMultiBody(baseMass=0,baseVisualShapeIndex=vis,basePosition=list(pos))
def _ring(pos, color=[1,.85,.2], R=0.6, n=24):
    for i in range(n):
        a0,a1=2*math.pi*i/n,2*math.pi*(i+1)/n
        p.addUserDebugLine([pos[0]+R*math.cos(a0),pos[1]+R*math.sin(a0),pos[2]],[pos[0]+R*math.cos(a1),pos[1]+R*math.sin(a1),pos[2]],color,1,lifeTime=0)
def _arrow(base, nd, sid, hli, hri):
    R=[1,0,0]; W=2; tip=np.array(base)+0.5*nd; up=np.array([0,0,1]); perp=np.cross(nd,up); mg=np.linalg.norm(perp); perp=perp/mg if mg>.01 else np.array([1,0,0])
    hb=tip-0.12*nd; lp=(hb+.07*perp).tolist(); rp=(hb-.07*perp).tolist(); tip=tip.tolist(); base=list(base)
    sid =p.addUserDebugLine(base,tip, R,W,lifeTime=0,**({'replaceItemUniqueId':sid}  if sid  is not None else {}))
    hli =p.addUserDebugLine(tip, lp,  R,W,lifeTime=0,**({'replaceItemUniqueId':hli}  if hli  is not None else {}))
    hri =p.addUserDebugLine(tip, rp,  R,W,lifeTime=0,**({'replaceItemUniqueId':hri}  if hri  is not None else {}))
    return sid,hli,hri

if HOVER_ONLY or FACE_ONLY: _sphere(env_raw.hover_target,.15,[1,1,0,.8])
sp0,_=p.getBasePositionAndOrientation(env_raw.drone_id); _sphere(list(sp0),.1,[.2,1,.1,1])
for g in env_raw.gold_data: _ring(g['pos'])
p.addUserDebugText(f"S{STAGE_TO_WATCH}|{RUN_NAME}|BEST",[0,0,4.6],textColorRGB=[.7,.7,.7],textSize=1.2,lifeTime=0)

sid=hli=hri=txt=prev_trail=None; ep_r=ep_s=0; ep_t=time.time(); step_r=0.
_tc=[g['pos'][:] for g in env_raw.gold_data]

while True:
    done=False
    for _ in range(RENDER_STRIDE):
        act,_=model.predict(obs,deterministic=True); obs,rews,dones,_=env.step(act)
        step_r=rews[0]; ep_r+=step_r; ep_s+=1
        cp=[g['pos'][:] for g in env_raw.gold_data]
        for pos in list(_tc):
            if pos not in cp: _ring(pos,[.55,.55,.55],R=.35); _tc.remove(pos)
        if dones[0]: done=True; break
    if done:
        print(f"Ep end | steps={ep_s} sim={ep_s/240:.1f}s R={ep_r:.1f}")
        ep_r=ep_s=0; step_r=0.; sid=hli=hri=txt=None; time.sleep(1.)
        p.removeAllUserDebugItems(); prev_trail=None
        if HOVER_ONLY or FACE_ONLY: _sphere(env_raw.hover_target,.15,[1,1,0,.8])
        sp,_=p.getBasePositionAndOrientation(env_raw.drone_id); _sphere(list(sp),.1,[.2,1,.1,1])
        for g in env_raw.gold_data: _ring(g['pos'])
        p.addUserDebugText(f"S{STAGE_TO_WATCH}|{RUN_NAME}|BEST",[0,0,4.6],textColorRGB=[.7,.7,.7],textSize=1.2,lifeTime=0)
        _tc=[g['pos'][:] for g in env_raw.gold_data]; ep_t=time.time()
        try:
            with open(vecnorm_path,'rb') as f: saved=cloudpickle.load(f)
            env.obs_rms=saved.obs_rms; env.ret_rms=saved.ret_rms; model=PPO.load(model_path,env=env)
        except: pass
    dp,ori=p.getBasePositionAndOrientation(env_raw.drone_id); eu=p.getEulerFromQuaternion(ori)
    rot=np.array(p.getMatrixFromQuaternion(ori)).reshape(3,3); nd=rot@np.array([0,1,0])
    sid,hli,hri=_arrow(dp,nd,sid,hli,hri)
    if ep_s%max(12,RENDER_STRIDE)==0:
        if prev_trail: p.addUserDebugLine(prev_trail,list(dp),[.2,1.,.1],2,lifeTime=0)
        prev_trail=list(dp)
    if FACE_ONLY:
        fv=np.array(env_raw.hover_target)-np.array(dp); fd=np.linalg.norm(fv)
        cos_t=(rot.T@fv)[1]/fd if fd>.05 else 0.
        lbl=f"cos={cos_t:.3f} streak={env_raw._face_streak}"
    elif HOVER_ONLY: lbl=f"dist={np.linalg.norm(np.array(dp)-np.array(env_raw.hover_target)):.2f}m"
    elif env_raw.gold_data: lbl=f"dist={min(math.sqrt(sum((a-b)**2 for a,b in zip(dp,g['pos']))) for g in env_raw.gold_data):.2f}m"
    else: lbl="done"
    tilt=math.degrees(math.sqrt(eu[0]**2+eu[1]**2)); et=time.time()-ep_t
    txt_s=f"{lbl} Tilt:{tilt:.1f}° T:{et:.0f}s r:{step_r:.3f} EpR:{ep_r:.1f} x{RENDER_STRIDE} [{RUN_NAME}]"
    tp=[dp[0],dp[1],dp[2]+1.2]
    if txt is None: txt=p.addUserDebugText(txt_s,tp,textColorRGB=[1,0,0],textSize=1.2,lifeTime=0)
    else: txt=p.addUserDebugText(txt_s,tp,textColorRGB=[1,0,0],textSize=1.2,lifeTime=0,replaceItemUniqueId=txt)